# Explore screener HTTP API

Call **screener** (validate rules, run screen, list presets).

**Prerequisites**
- Run screener with uvicorn; default **`http://localhost:8011`** (adjust with `SCREENER_BASE_URL`).
- Venv: `make venv-service SERVICE=screener`; `pip install jupyterlab ipykernel httpx` if needed.
- Server needs datalake connectivity per its config for full `/screen/run` flows.

**Endpoints:** `GET /health`, `GET /screen/presets`, `POST /screen/validate`.

Add more `.ipynb` files in this directory for ad-hoc experiments; keep `explore.ipynb` as the shared template.

In [ ]:
import json
import os

import httpx

BASE_URL = os.environ.get("SCREENER_BASE_URL", "http://localhost:8011").rstrip("/")
print("BASE_URL =", BASE_URL)

In [ ]:
with httpx.Client(timeout=60.0) as client:
    r = client.get(f"{BASE_URL}/health")
    r.raise_for_status()
    print(json.dumps(r.json(), indent=2))

In [ ]:
with httpx.Client(timeout=60.0) as client:
    r = client.get(f"{BASE_URL}/screen/presets")
    r.raise_for_status()
    presets = r.json()
    print(f"presets count: {len(presets)}")
    print(json.dumps(presets[:3], indent=2, default=str) if presets else "(empty)")

In [ ]:
payload = {
    "start_date": "2024-01-02",
    "end_date": "2024-06-28",
    "rules": {"price_range": {"min_price": 1.0, "max_price": 10000.0}},
}

with httpx.Client(timeout=60.0) as client:
    r = client.post(f"{BASE_URL}/screen/validate", json=payload)
    if r.status_code >= 400:
        print(r.status_code, r.text)
    else:
        r.raise_for_status()
        print(json.dumps(r.json(), indent=2))

## Next steps

- Build a `POST /screen/run` body using `ScreenRequest` from `app/models.py`.
- Try a preset bundle from `/screen/presets`.